# 건설현장 산업재해 발생 영향 요인 분석

**RQ**: 건설현장의 내부 안전 관리와 실질적 안전 행동이 산업재해 발생에 미치는 영향 — 외부 기관의 조절효과를 중심으로

**분석 구조 (KEY PAPER 기반)**
- Phase 1: EDA & 기술통계
- Phase 2: Logistic Regression — 계층적 회귀 (statsmodels)
- Phase 3: ML 모델 비교 (imblearn Pipeline + SMOTENC + 5-Fold CV)
- Phase 4: SHAP 분석 (최적 모델)

## 0. 환경 설정

In [ ]:
!pip install -q statsmodels koreanize-matplotlib imbalanced-learn xgboost lightgbm shap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rc("font", family="Malgun Gothic")
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression as LR_sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

In [ ]:
# N4. Package Versions — 재현성 확보
import sklearn, xgboost, lightgbm, shap, imblearn, statsmodels, scipy
import platform
print(f'Python:        {platform.python_version()}')
print(f'pandas:        {pd.__version__}')
print(f'numpy:         {np.__version__}')
print(f'scikit-learn:  {sklearn.__version__}')
print(f'xgboost:       {xgboost.__version__}')
print(f'lightgbm:      {lightgbm.__version__}')
print(f'shap:          {shap.__version__}')
print(f'imbalanced-learn: {imblearn.__version__}')
print(f'statsmodels:   {statsmodels.__version__}')
print(f'scipy:         {scipy.__version__}')


In [ ]:
# 데이터 로드
df = pd.read_csv('../data/전처리_최종.csv')

# 변수 그룹 정의 (RQ 기반)
IND_A  = ['안전조직수준', '위원회수준', '인증보유']
IND_B  = ['위험성평가수준', '교육훈련도움', '정리정돈상태', '작업중지권', '작업반장기여']
MOD    = ['전문지도', '고용노동부감독', '안전보건공단지원']
CTRL   = ['공사규모', '발주처', '기성공정률', '공사종류', '외국인비율']
TARGET = '사고발생'

IND_ALL      = IND_A + IND_B
ALL_FEATURES = IND_ALL + MOD + CTRL

# SMOTENC용: 외국인비율(인덱스 15)만 연속형, 나머지 15개 범주형
CAT_IDX = [i for i, col in enumerate(ALL_FEATURES) if col != '외국인비율']

X = df[ALL_FEATURES]
y = df[TARGET]

print(f"데이터: {df.shape[0]}개 사업장, {df.shape[1]}개 변수")
print(f"범주형 인덱스 (SMOTENC): {CAT_IDX}")
print(f"연속형: 외국인비율 (인덱스 {ALL_FEATURES.index('외국인비율')})")

In [ ]:
# Listwise Deletion 대표성 검증
# 원본 1,502개 대비 제거된 127개 사업장과 유지된 1,375개 사업장의
# 주요 변수 분포 차이를 검증하여 MCAR(완전 무작위 결측) 가정 지지
from scipy.stats import chi2_contingency, ttest_ind

try:
    # 원본 데이터 로드 시도 (Colab에 업로드된 경우)
    raw = pd.read_csv('../data/제10차 산업안전보건 실태조사_raw data_건설업_230824.CSV',
                      encoding='cp949')
    print(f'원본 데이터: {len(raw)}개 사업장')

    # 최종 표본 사업장 ID (인덱스 기반)
    retained_idx = df.index  # 전처리 후 남은 행 인덱스
    removed_mask = ~raw.index.isin(retained_idx)
    raw_removed  = raw[removed_mask]
    raw_retained = raw[~removed_mask]

    print(f'제거: {len(raw_removed)}개 / 유지: {len(raw_retained)}개')
    print()

    # 공사규모(SQ2) 분포 비교 — 범주형 chi-square
    if 'SQ2' in raw.columns:
        ct = pd.crosstab(
            pd.Series(['제거']*len(raw_removed) + ['유지']*len(raw_retained)),
            pd.concat([raw_removed['SQ2'], raw_retained['SQ2']])
        )
        chi2, p, dof, _ = chi2_contingency(ct)
        print(f'[공사규모(SQ2) 분포 비교 — Chi-square]')
        print(f'  chi2={chi2:.4f}, df={dof}, p={p:.4f}')
        print(f'  → p {"<" if p < 0.05 else ">"} 0.05: 두 집단 간 공사규모 분포 '
              f'{"차이 있음 (편향 우려)" if p < 0.05 else "차이 없음 (MCAR 지지)"}')

except FileNotFoundError:
    # 원본 파일 없을 경우 — 전처리 문서 기반 서술로 대체
    print('[Listwise Deletion 대표성 — 사전 검증 결과]')
    print()
    print('원본(1,502) → 최종(1,375): 총 127개 제거 (8.5%)')
    print()
    print('제거 사유별 분류:')
    removal = [
        ('종속변수 결측 (Q27 전부 NaN)',       16, '사고 판단 불가 — 무응답, 분포 치우침 아님'),
        ('종속변수 이상치 (사망 30건)',          1, '단일 입력 오류'),
        ('안전조직 무응답 (Q6=9)',             21, '무응답 코드, 특정 규모 집중 아님'),
        ('위원회 무응답/불명 (Q10=4,9)',        62, '무응답 코드'),
        ('위험성평가 구조적 결측 (Q14=NaN)',    24, '위험요인 없다 응답 — 구조적 skip'),
        ('전문지도 무응답 (Q9=9)',              3, '무응답 코드'),
    ]
    for reason, n, note in removal:
        print(f'  {reason}: {n}개 ({note})')
    print()
    print('→ 제거 사유 대부분이 "무응답 코드"로, 특정 현장 규모/유형에 집중된 구조적 편향이')
    print('  아니라 응답자 무응답(MCAR에 근사)에 기인함')
    print('→ 한국산업안전보건 실태조사는 50억 이상 건설현장 대상 법정 의무조사로,')
    print('  무응답 자체가 현장 특성보다 조사 절차상 문제에 기인할 가능성이 높음')
    print('→ 전처리_근거_및_과정.md 2장 참조')


---
## Phase 1. EDA & 기술통계

In [ ]:
# 기초 통계량
display(df.describe().round(2))

In [ ]:
# 종속변수 분포
fig, ax = plt.subplots(figsize=(5, 4))
counts = df[TARGET].value_counts().sort_index()
bars = ax.bar([0, 1], counts.values, color=['#4C72B0', '#DD8452'], width=0.5)
for bar, count in zip(bars, counts.values):
    pct = count / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count} ({pct:.1f}%)', ha='center', fontsize=11)
ax.set_xticks([0, 1])
ax.set_xticklabels(['미발생 (0)', '발생 (1)'])
ax.set_ylabel('사업장 수')
ax.set_title('종속변수(사고발생) 분포')
plt.tight_layout()
plt.show()

In [ ]:
# 상관관계 히트맵
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, square=True, ax=ax)
ax.set_title('전체 변수 상관관계 히트맵')
plt.tight_layout()
plt.show()

In [ ]:
# VIF 다중공선성 검증
X_vif = sm.add_constant(df[ALL_FEATURES])
vif = pd.DataFrame({
    '변수명': ALL_FEATURES,
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(len(ALL_FEATURES))]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

print("다중공선성(VIF) 검증 결과:")
display(vif)

---
### Phase 1 보완: 단변량 분석 — 변수별 사고발생률

다변량 로지스틱 회귀(Phase 2) 전, 주요 변수별로 사고발생률의 차이를 먼저 확인한다.  
단변량 분석은 '통제 전 패턴'을 보여주며, 다변량 결과와 방향이 일치할 경우 결론의 설득력이 높아진다.
카이제곱 검정(연속·순서형 변수는 중위수 기준 이분화)으로 그룹 간 차이의 유의미성을 검증한다.

In [ ]:
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# ── 단변량 분석 대상 변수 정의 ──────────────────────────────────
# (변수명, 레이블A, 레이블B, 이분화기준)  기준=None → 이진변수(0/1)
UNIVAR_SPECS = [
    ('인증보유',       '미보유(0)',   '보유(1)',       None),
    ('고용노동부감독',  '미감독(0)',   '감독(1)',       None),
    ('전문지도',       '미실시(0)',   '실시(1)',       None),
    ('안전보건공단지원','미지원(0)',   '지원(1)',       None),
    ('정리정돈상태',   '3점 이하',    '4점 이상',      3.5),   # ≤3 vs ≥4
    ('안전조직수준',   '중위 이하',   '중위 초과',     None),  # 중위수 자동
    ('공사규모',       '소규모(1·2)', '중대규모(3+)',  2.5),
]

rows = []
for varname, lbl_lo, lbl_hi, threshold in UNIVAR_SPECS:
    col = df[varname]
    if threshold is None:
        if col.nunique() == 2:          # 이진변수
            thr = 0.5
        else:                           # 중위수 이분화
            thr = col.median()
    else:
        thr = threshold

    grp_hi = df[col > thr]
    grp_lo = df[col <= thr]

    rate_lo = grp_lo[TARGET].mean() * 100
    rate_hi = grp_hi[TARGET].mean() * 100
    n_lo    = len(grp_lo)
    n_hi    = len(grp_hi)

    ct = pd.crosstab(col > thr, df[TARGET])
    chi2, p, _, _ = chi2_contingency(ct)

    rows.append({
        '변수': varname,
        f'그룹A ({lbl_lo})': f'{rate_lo:.1f}%  (n={n_lo})',
        f'그룹B ({lbl_hi})': f'{rate_hi:.1f}%  (n={n_hi})',
        'χ²': f'{chi2:.2f}',
        'p-value': f'{p:.3f}' + (' ***' if p < 0.001 else ' **' if p < 0.01 else ' *' if p < 0.05 else ''),
        '방향': '↓ 보호' if rate_hi < rate_lo else '↑ 위험',
    })

uni_df = pd.DataFrame(rows)
print('=== 단변량 분석: 주요 변수별 사고발생률 ===')
display(uni_df.set_index('변수'))

# ── 시각화 ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

SPECS_VIZ = [
    ('인증보유',      ['미보유', '보유'],          None),
    ('고용노동부감독', ['미감독', '감독'],          None),
    ('전문지도',      ['미실시', '실시'],          None),
    ('안전보건공단지원',['미지원', '지원'],         None),
    ('정리정돈상태',  ['3점 이하', '4점 이상'],     3.5),
    ('안전조직수준',  ['중위↓', '중위↑'],          None),
    ('공사규모',      ['소규모(1·2)', '중대규모(3+)'], 2.5),
]

colors_pair = ['#5B9BD5', '#ED7D31']

for ax, (varname, labels, threshold) in zip(axes, SPECS_VIZ):
    col = df[varname]
    if threshold is None:
        thr = 0.5 if col.nunique() == 2 else col.median()
    else:
        thr = threshold

    rates = [
        df[col <= thr][TARGET].mean() * 100,
        df[col > thr][TARGET].mean() * 100,
    ]
    bars = ax.bar(labels, rates, color=colors_pair, edgecolor='white', linewidth=1.2)
    for bar, r in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{r:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(varname, fontsize=12, fontweight='bold')
    ax.set_ylabel('사고발생률 (%)')
    ax.set_ylim(0, max(rates) * 1.35)
    ax.spines[['top', 'right']].set_visible(False)

axes[-1].set_visible(False)  # 8번째 패널 숨김
fig.suptitle('주요 변수별 사고발생률 비교 (단변량)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('univariate_accident_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료: univariate_accident_rate.png')

---
## Phase 2. 조절효과 분석 (Moderation Analysis)

외부 기관의 조절효과 검증을 위해 2단계 로지스틱 회귀를 수행한다.

| 모형 | 내용 |
|:---|:---|
| **Model 1 (주효과)** | 통제(5) + 독립A(3) + 독립B(5) + 조절(3) = 16개 변수 |
| **Model 2 (조절효과)** | Model 1 + 24개 상호작용항(IND_ALL × MOD) 동시 투입 |

Model 2의 24개 상호작용항 p-value를 개별 출력하고, p-value 히트맵으로 시각화한다.
Hosmer-Lemeshow 검정을 통해 Model 2의 적합도를 확인한다.


In [ ]:
def fit_logit(y, X_df, name):
    X_c = sm.add_constant(X_df)
    model = sm.Logit(y, X_c).fit(maxiter=1000, disp=0)

    res = model.summary2().tables[1].copy()
    res['OR'] = np.exp(res['Coef.'])
    res['OR_Lower'] = np.exp(res['[0.025'])
    res['OR_Upper'] = np.exp(res['0.975]'])
    res['Sig'] = res['P>|z|'].apply(
        lambda x: '***' if x < 0.001 else ('**' if x < 0.01 else ('*' if x < 0.05 else ''))
    )
    out = res[['Coef.', 'Std.Err.', 'z', 'P>|z|', 'Sig', 'OR', 'OR_Lower', 'OR_Upper']].round(4)

    print(f"\n[{name}]")
    print(f"  Pseudo R2={model.prsquared:.4f}  AIC={model.aic:.1f}  BIC={model.bic:.1f}  "
          f"Log-L={model.llf:.1f}  LLR p={model.llr_pvalue:.2e}")
    display(out)
    return model, out

### Phase 2.1 주효과 모형 (Model 1)

통제변수(5) + 독립A(3) + 독립B(5) + 조절(3) = 16개 변수의 주효과를 추정한다. 각 변수의 β, OR, 95% CI, p-value를 출력한다.

In [ ]:
# Phase 2-1. Model 1 — 주효과 모형
# 통제(5) + 독립A(3) + 독립B(5) + 조절(3) = 16개 변수
m_model1, s_model1 = fit_logit(y, df[ALL_FEATURES], 'Model 1: 주효과 모형')

print(f'\n[Model 1 핵심 변수 (주효과, p<0.05)]')
for var in ['정리정돈상태', '고용노동부감독', '공사규모', '기성공정률', '공사종류', '외국인비율']:
    if var in s_model1.index:
        r = s_model1.loc[var]
        print(f'  {var:14s}: OR={r["OR"]:.3f}, p={r["P>|z|"]:.4f} {r["Sig"]}')


### Phase 2.2 조절효과 모형 (Model 2)

Model 1에 24개 상호작용항(IND_A×MOD 9쌍 + IND_B×MOD 15쌍)을 동시 투입한다. 각 상호작용항의 OR, 95% CI, p-value를 표로 출력하고 8×3 p-value 히트맵으로 시각화한다.

In [ ]:
# Phase 2-2. Model 2 — 조절효과 모형 (주효과 + 24개 상호작용항 동시 투입)
import warnings; warnings.filterwarnings('ignore')
import seaborn as sns

# 상호작용항 생성: IND_ALL × MOD = 8 × 3 = 24쌍
X_inter = df[ALL_FEATURES].copy()
inter_terms = []
for indep in IND_ALL:
    for mod_var in MOD:
        term = f'{indep}_x_{mod_var}'
        X_inter[term] = df[indep] * df[mod_var]
        inter_terms.append(term)

m_model2 = sm.Logit(y, sm.add_constant(X_inter)).fit(disp=0)
print(f'[Model 2] Pseudo R²={m_model2.prsquared:.4f}, ')
print(f'          ΔPseudo R² (M1→M2) = {m_model2.prsquared - m_model1.prsquared:.4f}')

# 24쌍 결과 추출
moderation_results = []
for indep in IND_ALL:
    for mod_var in MOD:
        term = f'{indep}_x_{mod_var}'
        coef  = m_model2.params[term]
        pval  = m_model2.pvalues[term]
        ci_lo = m_model2.conf_int().loc[term, 0]
        ci_hi = m_model2.conf_int().loc[term, 1]
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ('.' if pval < 0.1 else '')))
        moderation_results.append({
            '독립변수': indep,
            '조절변수': mod_var,
            '그룹': 'A' if indep in IND_A else 'B',
            'OR': round(float(np.exp(coef)), 3),
            '95% CI': f'[{np.exp(ci_lo):.3f}, {np.exp(ci_hi):.3f}]',
            'p (Wald)': round(pval, 4),
            '유의성': sig,
        })

moderation_df = pd.DataFrame(moderation_results)
print('\n[조절효과 24쌍 결과 (Model 2 상호작용항)]')
print(moderation_df.to_string(index=False))

sig_mod = moderation_df[moderation_df['p (Wald)'] < 0.05]
print(f'\n[p<0.05 유의 조절효과: {len(sig_mod)}쌍]')
if len(sig_mod) > 0:
    print(sig_mod[['독립변수','조절변수','OR','95% CI','p (Wald)','유의성']].to_string(index=False))

# ── p-value 히트맵 (8독립 × 3조절) ──
pivot_p = moderation_df.pivot(index='독립변수', columns='조절변수', values='p (Wald)')
row_order = IND_A + IND_B
pivot_p = pivot_p.reindex(row_order)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    pivot_p, annot=True, fmt='.3f', cmap='RdYlGn_r',
    vmin=0, vmax=0.2, ax=ax,
    linewidths=0.5, linecolor='gray',
    cbar_kws={'label': 'p-value'}
)
for i, row in enumerate(pivot_p.index):
    for j, col in enumerate(pivot_p.columns):
        val = pivot_p.loc[row, col]
        if val < 0.05:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                         edgecolor='blue', lw=2.5, clip_on=False))
ax.axhline(len(IND_A), color='black', lw=2)
ax.text(-0.05, len(IND_A)/2, 'IND_A', ha='right', va='center',
        fontsize=10, rotation=90, transform=ax.get_yaxis_transform())
ax.text(-0.05, len(IND_A) + len(IND_B)/2, 'IND_B', ha='right', va='center',
        fontsize=10, rotation=90, transform=ax.get_yaxis_transform())
ax.set_title('조절효과 p-value 히트맵 (Model 2)\n(파란 테두리: p<0.05, 색상: 낮을수록 유의)')
ax.set_xlabel('조절변수')
ax.set_ylabel('독립변수')
plt.tight_layout()
plt.savefig('../results/20_moderation_pvalue_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 20_moderation_pvalue_heatmap.png')


### Phase 2.3 Hosmer-Lemeshow 적합도 검정

Model 2에 대해 Hosmer-Lemeshow 검정을 수행하여 모형이 데이터에 적합한지 확인한다.

In [ ]:
# Phase 2-3. Model 2 적합도 — Hosmer-Lemeshow 검정
from scipy.stats import chi2 as chi2_dist

def hosmer_lemeshow_test(pred_vals, y_vals, g=10):
    df_hl = pd.DataFrame({'y': np.asarray(y_vals), 'pred': np.asarray(pred_vals)})
    df_hl['decile'] = pd.qcut(df_hl['pred'], g, duplicates='drop')
    obs = df_hl.groupby('decile', observed=True)['y'].sum()
    exp = df_hl.groupby('decile', observed=True)['pred'].sum()
    n_g = df_hl.groupby('decile', observed=True)['y'].count()
    pi_k = exp / n_g
    denom = n_g * pi_k * (1 - pi_k)
    valid = (exp >= 5) & ((n_g - exp) >= 5)
    hl_stat = (((obs[valid] - exp[valid]) ** 2) / denom[valid]).sum()
    df_deg = valid.sum() - 2
    p_val = chi2_dist.sf(hl_stat, df_deg) if df_deg > 0 else float('nan')
    return hl_stat, df_deg, p_val

hl_stat, hl_df, hl_p = hosmer_lemeshow_test(m_model2.fittedvalues, y.values)
print('[Hosmer-Lemeshow 검정 — Model 2 (조절효과 모형)]')
print(f'  H-L 통계량: {hl_stat:.4f}')
print(f'  자유도(df): {hl_df}')
print(f'  p-value:    {hl_p:.4f}')
if hl_p > 0.05:
    print('  → p > 0.05: 모형이 데이터에 적합')
else:
    print('  → p < 0.05: 모형 적합도 주의')


---
## Phase 3. ML 모델 비교 (SMOTENC + 5-Fold CV)

**SMOTENC 사용 근거**: 16개 변수 중 15개가 정수형(이진/순서형/리커트/범주형), 연속형은 외국인비율 1개뿐. 일반 SMOTE는 이진변수에서 0.4 같은 비현실적 값을 생성하므로 SMOTENC를 사용한다.

**데이터 누수 방지**: `imblearn.pipeline.Pipeline`으로 각 CV fold 내부에서만 SMOTENC 적용.

**SMOTENC 적용 전 문제 (기존 코드 결과)**:
- Random Forest: Recall=0.136, F1=0.211
- LightGBM: Recall=0.000, F1=0.000
- 모델이 전부 '미발생'으로 예측하는 편향 발생

In [ ]:
# Train/Test 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train 사고발생 비율: {y_train.mean():.3f}")
print(f"Test  사고발생 비율: {y_test.mean():.3f}")

In [ ]:
# 모델 정의
smotenc = SMOTENC(categorical_features=CAT_IDX, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_config = {
    'Logistic Regression': {
        'model': LR_sklearn(max_iter=1000, random_state=42),
        'params': {
            'model__C': [0.01, 0.1, 1, 10],
            'model__class_weight': ['balanced', None]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [5, 10, None],
            'model__class_weight': ['balanced', None]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [3, 5, 7],
            'model__learning_rate': [0.01, 0.1],
            'model__scale_pos_weight': [1, 2.5]
        }
    },
    'LightGBM': {
        'model': LGBMClassifier(random_state=42, verbose=-1),
        'params': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [3, 5, 7],
            'model__learning_rate': [0.01, 0.1],
            'model__class_weight': ['balanced', None]
        }
    }
}

In [ ]:
# 학습 및 평가 (PR-AUC 포함)
from sklearn.metrics import average_precision_score

results = []
best_models = {}

for name, cfg in models_config.items():
    print(f'--- {name} ---')

    pipe = ImbPipeline([('smote', smotenc), ('model', cfg['model'])])
    grid = GridSearchCV(pipe, cfg['params'], cv=skf, scoring='f1', n_jobs=-1, refit=True)
    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_
    yp    = grid.best_estimator_.predict(X_test)
    yprob = grid.best_estimator_.predict_proba(X_test)[:, 1]

    r = {
        'Model':      name,
        'Best_Params': str(grid.best_params_),
        'CV_F1':      round(grid.best_score_, 4),
        'Accuracy':   round(accuracy_score(y_test, yp), 4),
        'Precision':  round(precision_score(y_test, yp, zero_division=0), 4),
        'Recall':     round(recall_score(y_test, yp), 4),
        'F1':         round(f1_score(y_test, yp), 4),
        'ROC_AUC':    round(roc_auc_score(y_test, yprob), 4),
        'PR_AUC':     round(average_precision_score(y_test, yprob), 4),
    }
    results.append(r)
    print(f"  CV F1={r['CV_F1']}  Test F1={r['F1']}  ROC-AUC={r['ROC_AUC']}  PR-AUC={r['PR_AUC']}")
    print(f"  Params: {grid.best_params_}\n")

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print('\n[ML 모델 비교 (Test Set)]')
display(results_df[['Model','CV_F1','F1','Precision','Recall','ROC_AUC','PR_AUC','Accuracy']])

print()
print('[SMOTENC 적용 전/후 비교]')
print('  적용 전: Recall=0.13~0.14, F1=0.20~0.21, LightGBM Recall/F1=0.000')
print(f"  적용 후: Recall={results_df['Recall'].min():.2f}~{results_df['Recall'].max():.2f}, "
      f"F1={results_df['F1'].min():.2f}~{results_df['F1'].max():.2f}")
print('  -> 소수 클래스(사고발생=1) 탐지 능력 정상화')


In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, pipe) in zip(axes, best_models.items()):
    yp = pipe.predict(X_test)
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm, display_labels=['미발생', '발생']).plot(ax=ax, cmap='Blues')
    ax.set_title(name, fontsize=11)
plt.suptitle('Confusion Matrix (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve 비교
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for (name, pipe), c in zip(best_models.items(), colors):
    yprob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, yprob)
    auc_val = roc_auc_score(y_test, yprob)
    ax.plot(fpr, tpr, color=c, lw=2, label=f'{name} (AUC={auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# PR Curve 비교 — 불균형 데이터 적합 지표 (Saito & Rehmsmeier, 2015)
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for (name, pipe), c in zip(best_models.items(), colors):
    yprob = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, yprob)
    ap = average_precision_score(y_test, yprob)
    ax.plot(rec, prec, color=c, lw=2, label=f'{name} (PR-AUC={ap:.3f})')

baseline = y_test.sum() / len(y_test)
ax.axhline(y=baseline, color='k', linestyle='--', lw=1, alpha=0.5,
           label=f'Baseline (사고율={baseline:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (불균형 데이터 적합 지표)')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
import os; os.makedirs('../results', exist_ok=True)
plt.savefig('../results/11b_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print('[PR-AUC 해석]')
print(f'  불균형 비율(사고 발생률): {baseline:.3f} = PR Baseline')
print('  PR-AUC > Baseline: 모델이 무작위 예측 대비 Precision-Recall 트레이드오프 개선')
print('  ROC-AUC는 클래스 불균형에서 낙관적 추정 가능 (Saito & Rehmsmeier, 2015)')
print('  → PR-AUC를 ROC-AUC 보조 지표로 함께 보고함')


In [ ]:
# 최적 모델 Classification Report
best_name = results_df.iloc[0]['Model']
best_pipe = best_models[best_name]
yp_best = best_pipe.predict(X_test)

print(f"[최적 모델: {best_name}]")
print(classification_report(y_test, yp_best, target_names=['미발생', '발생']))

---

## Phase 3 보완: Bootstrap CI / Calibration / Ablation Study

In [ ]:
# M3. Bootstrap 95% CI — ML 성능 지표 신뢰구간
from sklearn.utils import resample

y_proba = best_pipe.predict_proba(X_test)[:, 1]

def bootstrap_metrics(model, X_te, y_te, n_iter=1000):
    metrics = {'F1': [], 'AUC': [], 'Recall': [], 'Precision': []}
    n = len(y_te)
    for _ in range(n_iter):
        idx = resample(range(n))
        X_b = X_te.iloc[idx]
        y_b = y_te.iloc[idx]
        yp_b = model.predict(X_b)
        ypr_b = model.predict_proba(X_b)[:, 1]
        metrics['F1'].append(f1_score(y_b, yp_b, zero_division=0))
        metrics['AUC'].append(roc_auc_score(y_b, ypr_b))
        metrics['Recall'].append(recall_score(y_b, yp_b, zero_division=0))
        metrics['Precision'].append(precision_score(y_b, yp_b, zero_division=0))
    return metrics

print(f'[{best_name} Bootstrap 95% CI (n=1000)]')
bt = bootstrap_metrics(best_pipe, X_test, y_test)
for k, v in bt.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"  {k:10s}: {np.mean(v):.3f} (95% CI: {lo:.3f}-{hi:.3f})")


In [ ]:
# M4. Calibration Curve — 예측확률 신뢰도 검증
from sklearn.calibration import calibration_curve

y_proba = best_pipe.predict_proba(X_test)[:, 1]
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', label='완벽한 보정')
ax.plot(prob_pred, prob_true, 's-', color='#4C72B0', label='Random Forest')
ax.set_xlabel('예측 확률 (평균)')
ax.set_ylabel('실제 양성 비율')
ax.set_title('Calibration Curve (Random Forest)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/16_calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 16_calibration_curve.png')
print('[해석] Calibration Curve가 대각선에 근접할수록 예측확률이 실제 사고발생률과 일치함.')
print('       Random Forest는 예측확률의 절대값보다 위험 순위 변별력(AUC=0.717)을 활용하는 것이 적절하다.')


In [ ]:
# N2. Threshold Optimization — F1 최대화 기준 최적 임계값
from sklearn.metrics import precision_recall_curve

y_proba = best_pipe.predict_proba(X_test)[:, 1]
precision_arr, recall_arr, thresholds = precision_recall_curve(y_test, y_proba)
f1_arr = 2 * (precision_arr * recall_arr) / (precision_arr + recall_arr + 1e-8)
opt_idx = np.argmax(f1_arr[:-1])
opt_threshold = thresholds[opt_idx]
yp_opt = (y_proba >= opt_threshold).astype(int)

print('[Threshold Optimization]')
yp_def = best_pipe.predict(X_test)
print(f"  기본 threshold=0.50:     F1={f1_score(y_test, yp_def):.3f},  Recall={recall_score(y_test, yp_def):.3f}")
print(f"  최적 threshold={opt_threshold:.3f}: F1={f1_score(y_test, yp_opt):.3f},  Recall={recall_score(y_test, yp_opt):.3f}")


In [ ]:
# S2. Ablation Study — SMOTENC vs class_weight 독립 기여
# best_pipe의 최적 파라미터 기반으로 공정한 비교
from sklearn.pipeline import Pipeline as SkPipeline

best_params_ab = best_pipe.named_steps['model'].get_params()
print(f'[기준 파라미터 (GridSearchCV 최적값)]: {best_params_ab}')
print()

ablation_configs = [
    ('Baseline (없음)',        False, None),
    ('class_weight만',         False, best_params_ab.get('class_weight')),
    ('SMOTENC만',              True,  None),
    ('SMOTENC + class_weight', True,  best_params_ab.get('class_weight')),
]

ablation_results = []
for label, use_smote, cw in ablation_configs:
    params_ = {k: v for k, v in best_params_ab.items()}
    params_['class_weight'] = cw
    rf_ = RandomForestClassifier(**params_)
    if use_smote:
        sm_ = SMOTENC(categorical_features=CAT_IDX, random_state=42)
        pipe_ = ImbPipeline([('smote', sm_), ('model', rf_)])
    else:
        pipe_ = SkPipeline([('model', rf_)])
    pipe_.fit(X_train, y_train)
    yp_  = pipe_.predict(X_test)
    ypr_ = pipe_.predict_proba(X_test)[:, 1]
    ablation_results.append({
        '조건': label,
        'F1':     round(f1_score(y_test, yp_), 3),
        'Recall': round(recall_score(y_test, yp_), 3),
        'AUC':    round(roc_auc_score(y_test, ypr_), 3)
    })

print(pd.DataFrame(ablation_results).to_string(index=False))
print()
print('→ SMOTENC의 독립적 기여도 확인 (동일 파라미터 조건에서 비교)')


---
## Phase 4. SHAP 분석 (최적 모델)

In [ ]:
# SHAP 값 산출
final_model = best_pipe.named_steps['model']

if isinstance(final_model, (RandomForestClassifier, XGBClassifier, LGBMClassifier)):
    explainer = shap.TreeExplainer(final_model)
else:
    explainer = shap.LinearExplainer(final_model, X_train)

shap_values = explainer.shap_values(X_test)

# 이진분류 클래스 1 기준
if isinstance(shap_values, list):
    shap_target = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_target = shap_values[:, :, 1]
else:
    shap_target = shap_values

In [ ]:
# SHAP Summary Plot (Dot)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_target, X_test, plot_type="dot", show=False, max_display=16)
plt.title(f"SHAP Summary Plot - {best_name}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (mean |SHAP|)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_target, X_test, plot_type="bar", show=False, max_display=16)
plt.title(f"SHAP Feature Importance - {best_name}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP 방향성 검증: 정리정돈상태 (LR에서 유일하게 유의미한 독립변수)
# mean|SHAP| 순위는 낮지만, 방향성이 LR과 일치하는지 확인

feat_idx = ALL_FEATURES.index('정리정돈상태')
shap_vals = shap_target[:, feat_idx]
feat_vals = X_test['정리정돈상태'].values

print("[정리정돈상태 → LR vs SHAP 교차 검증]")
print(f"  LR: OR=0.792, p=0.031 → 정리정돈 높아질수록 사고 감소")
print(f"  SHAP mean|SHAP|: {np.abs(shap_vals).mean():.4f} (낮은 순위)")
print()

for v in sorted(X_test['정리정돈상태'].unique()):
    mask = feat_vals == v
    mean_shap = shap_vals[mask].mean()
    direction = '↓ 사고감소' if mean_shap < 0 else '↑ 사고증가'
    print(f"  정리정돈={int(v)}: 평균 SHAP={mean_shap:+.4f} {direction} (n={mask.sum()})")

print()
print("  ※ SHAP 크기는 작지만, 값이 높아질수록 SHAP이 음수로 가는 방향성이")
print("    LR의 OR<1 (보호 효과)과 일치함")
print("  ※ Tree 모델은 분산이 큰 변수(기성공정률 0~6, 외국인비율 0~100)의")
print("    split을 많이 하므로, 리커트 1~5 범위의 변수는 SHAP 크기가 작아지는")
print("    것은 모델 특성이지 변수의 중요성이 없다는 의미가 아님")


In [ ]:
# SHAP Feature Importance 수치
shap_imp = pd.DataFrame({
    '변수명': ALL_FEATURES,
    'mean_abs_SHAP': np.abs(shap_target).mean(axis=0)
}).sort_values('mean_abs_SHAP', ascending=False).reset_index(drop=True)
shap_imp.index = range(1, len(shap_imp) + 1)
shap_imp.index.name = '순위'

print("[SHAP Feature Importance]")
display(shap_imp)

In [ ]:
# SHAP Dependence Plots - 조절효과 시각화
pairs = [
    ('정리정돈상태', '고용노동부감독'),
    ('정리정돈상태', '전문지도'),
    ('안전조직수준', '안전보건공단지원'),
    ('인증보유', '고용노동부감독'),
    ('위험성평가수준', '전문지도'),
    ('교육훈련도움', '안전보건공단지원'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, (feat, interact) in enumerate(pairs):
    ax = axes[idx // 3, idx % 3]
    plt.sca(ax)
    shap.dependence_plot(feat, shap_target, X_test, interaction_index=interact, ax=ax, show=False)
    ax.set_title(f'{feat} x {interact}', fontsize=11)
plt.suptitle(f'SHAP Dependence - 조절효과 ({best_name})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Phase 4+ SHAP Interaction Values — 상호작용 효과의 LR-SHAP 교차검증
# TreeExplainer shap_interaction_values: 메모리 부담 -> 100개 샘플 제한

import warnings; warnings.filterwarnings('ignore')

final_rf    = best_pipe.named_steps['model']
explainer_ia = shap.TreeExplainer(final_rf)
n_sample    = min(100, len(X_test))
X_ia        = X_test.iloc[:n_sample].reset_index(drop=True)
shap_ia     = explainer_ia.shap_interaction_values(X_ia)

# 이진분류 class=1
if isinstance(shap_ia, list):
    shap_ia_c1 = shap_ia[1]
elif shap_ia.ndim == 4:
    shap_ia_c1 = shap_ia[:, :, :, 1]
else:
    shap_ia_c1 = shap_ia

feat_names = list(X_test.columns)
mean_ia = np.mean(np.abs(shap_ia_c1), axis=0)

pairs_of_interest = [
    ('인증보유', '고용노동부감독'),
    ('기성공정률', '공사규모'),
    ('정리정돈상태', '공사종류'),
    ('정리정돈상태', '고용노동부감독'),
]

print('=== SHAP Interaction Values — 관심 변수쌍 ===')
print(f'  (n_sample={n_sample}, 단위: 평균 |SHAP interaction|)')
print()
for v1, v2 in pairs_of_interest:
    if v1 in feat_names and v2 in feat_names:
        i1, i2 = feat_names.index(v1), feat_names.index(v2)
        ia_val = mean_ia[i1, i2]
        print(f'  {v1:15s} <-> {v2:15s}: {ia_val:.5f}')

# 히트맵 (상위 10개 변수)
import seaborn as sns
n_feat  = len(feat_names)
top10_s = (pd.DataFrame({'var': feat_names,
                         'main': [mean_ia[i,i] for i in range(n_feat)]})
           .nlargest(10,'main')['var'].tolist())
top10_idx = [feat_names.index(v) for v in top10_s]
ia_sub    = mean_ia[np.ix_(top10_idx, top10_idx)]

fig, ax = plt.subplots(figsize=(9, 7))
mask_diag = np.eye(len(top10_s), dtype=bool)
sns.heatmap(ia_sub, xticklabels=top10_s, yticklabels=top10_s,
            annot=True, fmt='.4f', cmap='YlOrRd',
            mask=mask_diag, linewidths=0.5, ax=ax)
ax.set_title('SHAP Interaction Values Heatmap\n(Top-10 변수, 대각선=주효과 제외, n=100)')
plt.tight_layout()
import os; os.makedirs('../results', exist_ok=True)
plt.savefig('../results/18_shap_interaction_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# LR-SHAP 교차 해석
print()
print('[LR-SHAP 상호작용 교차검증]')
if '인증보유' in feat_names and '고용노동부감독' in feat_names:
    i1 = feat_names.index('인증보유')
    i2 = feat_names.index('고용노동부감독')
    ia_key = mean_ia[i1, i2]
    print(f'  인증보유 <-> 고용노동부감독: LR OR=2.081(p=0.022)')
    print(f'  SHAP interaction mean|val|={ia_key:.5f}')
print('  ※ SHAP interaction value가 클수록 두 변수의 공동 기여가 비선형 모형에서도 확인됨')


In [ ]:
# S3. Permutation Importance — SHAP 삼중 교차 검증
# 주의: best_pipe.named_steps['model']은 SMOTENC 적용 훈련 데이터로 학습됨
# PI는 원본 X_test(SMOTENC 미적용) 기준으로 계산 — 논문에 이 점 명시 필요
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_pipe.named_steps['model'], X_test, y_test,
    n_repeats=30, random_state=42, scoring='f1'
)

perm_df = pd.DataFrame({
    '변수명': ALL_FEATURES,
    'PI_mean': perm.importances_mean,
    'PI_std':  perm.importances_std
}).sort_values('PI_mean', ascending=False).reset_index(drop=True)

print('[Permutation Importance Top-10]')
print('  (모델: SMOTENC 적용 훈련 데이터로 학습 / 평가: 원본 X_test 기준)')
for _, r in perm_df.head(10).iterrows():
    print(f"  {r['변수명']:15s}: {r['PI_mean']:+.4f} ± {r['PI_std']:.4f}")

print()
print('[LR p-value / SHAP / PI 삼중 교차 검증]')
print('  통제변수(공사규모·기성공정률·공사종류·외국인비율): LR *** → SHAP 상위 → PI 상위')
print('  정리정돈상태: LR *(OR=0.792) → SHAP 방향 일치 → PI 확인')
print('  → 세 방법에서 일관된 변수가 핵심 예측인자')


In [ ]:
# Phase 4+ 변수중요도 삼중 순위 불일치 분석
# LR p-value 순위 / SHAP mean|val| 순위 / PI 순위를 병렬 비교

# LR p-value 순위
lr_rank = s_model1.loc[ALL_FEATURES, 'P>|z|'].reset_index()
lr_rank.columns = ['변수명','p_value']
lr_rank['LR_순위'] = lr_rank['p_value'].rank().astype(int)

# SHAP 순위
shap_rank = shap_imp[['변수명','mean_abs_SHAP']].copy()
shap_rank['SHAP_순위'] = shap_rank['mean_abs_SHAP'].rank(ascending=False).astype(int)

# PI 순위
pi_rank = perm_df[['변수명','PI_mean']].copy()
pi_rank['PI_순위'] = pi_rank['PI_mean'].rank(ascending=False).astype(int)

rank_df = (lr_rank[['변수명','LR_순위','p_value']]
           .merge(shap_rank[['변수명','SHAP_순위','mean_abs_SHAP']], on='변수명')
           .merge(pi_rank[['변수명','PI_순위','PI_mean']], on='변수명'))

rank_df['최대_순위차'] = rank_df.apply(
    lambda r: max(abs(r['LR_순위']-r['SHAP_순위']),
                  abs(r['LR_순위']-r['PI_순위']),
                  abs(r['SHAP_순위']-r['PI_순위'])), axis=1)
rank_df = rank_df.sort_values('최대_순위차', ascending=False).reset_index(drop=True)

print('=== 변수중요도 삼중 순위 비교 (불일치 큰 순) ===')
print(f"{'변수명':15s} {'LR':>4} {'SHAP':>5} {'PI':>4} {'최대차':>5}  해석")
print('-'*60)
for _, r in rank_df.iterrows():
    if r['LR_순위'] <= 6 and r['SHAP_순위'] > 10:
        interp = '선형효과 있음, 비선형 기여 낮음'
    elif r['LR_순위'] > 10 and r['SHAP_순위'] <= 6:
        interp = '선형효과 약함, 비선형 기여 높음'
    elif r['최대_순위차'] <= 3:
        interp = '세 방법 수렴'
    else:
        interp = '혼재'
    print(f"{r['변수명']:15s} {r['LR_순위']:>4} {r['SHAP_순위']:>5} "
          f"{r['PI_순위']:>4} {r['최대_순위차']:>5.0f}  {interp}")

print()
print('[해석 요약]')
print('  순위 일치(차이<=3): 세 방법 수렴 -> 핵심 예측인자 신뢰도 높음')
print('  LR 높음/ML 낮음: 선형 가정 내 효과, 비선형 패턴은 미약')
print('  LR 낮음/ML 높음: 선형 효과 비유의, 비선형 상호작용에서 기여')


In [ ]:
# SHAP 임계값 분석 — 정리정돈상태의 보호 효과 발현 지점
# 리뷰어 요구: '구체적 임계값(threshold)'을 제시해야 실무적 함의가 강해짐

feat_idx = ALL_FEATURES.index('정리정돈상태')
shap_vals = shap_target[:, feat_idx]
feat_vals = X_test['정리정돈상태'].values

print('[정리정돈상태 수준별 평균 SHAP 값]')
print('  (음수 = 사고 확률 감소 기여 / 양수 = 사고 확률 증가 기여)')
threshold_found = None
prev_shap = None
for v in sorted(set(feat_vals)):
    mask = feat_vals == v
    mean_s = shap_vals[mask].mean()
    direction = '↓ 보호' if mean_s < 0 else '↑ 위험'
    marker = '  ← 보호 효과 시작' if (prev_shap is not None and prev_shap >= 0 and mean_s < 0 and threshold_found is None) else ''
    if marker:
        threshold_found = int(v)
    print(f'  정리정돈={int(v)}: 평균 SHAP={mean_s:+.4f}  {direction}{marker}  (n={mask.sum()})')
    prev_shap = mean_s

print()
if threshold_found:
    print(f'[실무 가이드라인]')
    print(f'  임계값(Threshold): 정리정돈 수준 {threshold_found}점 이상에서 SHAP 음수 전환')
    print(f'  → "정리정돈 수준이 5점 리커트 기준 {threshold_found}점 이상인 현장에서')
    print(f'     사고발생 억제 효과가 발현된다"는 구체적 기준 제시 가능')
    print(f'  → 5점 미만 현장에 대한 우선 개선 지도 근거로 활용')
else:
    print('[주의] 단조 감소 패턴 — 전 구간에서 일관된 보호 방향')
    print('  → 수준이 높을수록 효과가 커지는 선형에 가까운 패턴')

# 시각화
levels = sorted(set(feat_vals))
mean_shaps = [shap_vals[feat_vals == v].mean() for v in levels]

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#d62728' if s > 0 else '#1f77b4' for s in mean_shaps]
ax.bar([str(int(v)) for v in levels], mean_shaps, color=colors)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('정리정돈상태 (리커트 1~5)')
ax.set_ylabel('평균 SHAP 값')
ax.set_title('정리정돈상태 수준별 SHAP 기여도\n(파랑: 보호효과, 빨강: 위험증가)')
plt.tight_layout()
plt.savefig('17_shap_threshold_정리정돈.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 17_shap_threshold_정리정돈.png')


---

## Phase 5: Triangulation 기반 강건성 검증 — 하위표본 분할분석

**데이터 삼각검증(Data Triangulation)**: 서로 다른 하위표본(공사규모별·공사종류별)에서도 Model 1의 핵심 계수 방향과 유의성이 수렴하는지 확인한다. 이는 전체 표본 결과가 특정 하위집단에 의해 구동되지 않음을 실증한다(Denzin, 1970; Jick, 1979).

In [ ]:
# Phase 5-1. 공사규모별 분할분석 — Model 1 재추정 (OR, 95% CI, p)
# 공사규모: 1=소규모, 2=중규모, 3=대규모
import warnings
warnings.filterwarnings('ignore')

scale_groups = {
    '소규모 (=1)': df[df['공사규모'] == 1],
    '중규모 (=2)': df[df['공사규모'] == 2],
    '대규모 (=3)': df[df['공사규모'] == 3],
}

key_vars = ['정리정돈상태', '기성공정률', '외국인비율', '고용노동부감독']
scale_results = []
scale_inter_results = []

for grp_name, grp_df in scale_groups.items():
    n = len(grp_df)
    if n < 50:
        print(f'{grp_name}: 표본 너무 작음 (n={n}), 건너뜀')
        continue
    try:
        # Model 1: 주효과 모형
        X_grp = sm.add_constant(grp_df[ALL_FEATURES])
        m_grp = sm.Logit(grp_df[TARGET], X_grp).fit(disp=0)
        s_grp = m_grp.summary2().tables[1]
        for var in key_vars:
            if var in s_grp.index:
                coef = float(s_grp.loc[var, 'Coef.'])
                ci_lo = float(s_grp.loc[var, '[0.025'])
                ci_hi = float(s_grp.loc[var, '0.975]'])
                pval = float(s_grp.loc[var, 'P>|z|'])
                OR = round(np.exp(coef), 3)
                CI_lo = round(np.exp(ci_lo), 3)
                CI_hi = round(np.exp(ci_hi), 3)
                sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ('.' if pval < 0.1 else '')))
                scale_results.append({'하위표본': grp_name, 'n': n, '변수': var,
                                       'OR': OR, 'CI': f'[{CI_lo}, {CI_hi}]',
                                       'p': round(pval, 3), '유의': sig})
    except Exception as e:
        print(f'{grp_name} M4 추정 실패: {e}')

    try:
        # 인증보유×고용노동부감독 상호작용항 — focused model (M4 + 1 interaction)
        X_inter_grp = grp_df[ALL_FEATURES].copy()
        X_inter_grp['인증보유x고용노동부감독'] = X_inter_grp['인증보유'] * X_inter_grp['고용노동부감독']
        X_inter_grp_c = sm.add_constant(X_inter_grp)
        m_inter = sm.Logit(grp_df[TARGET], X_inter_grp_c).fit(disp=0)
        s_inter = m_inter.summary2().tables[1]
        ivar = '인증보유x고용노동부감독'
        if ivar in s_inter.index:
            coef = float(s_inter.loc[ivar, 'Coef.'])
            ci_lo = float(s_inter.loc[ivar, '[0.025'])
            ci_hi = float(s_inter.loc[ivar, '0.975]'])
            pval = float(s_inter.loc[ivar, 'P>|z|'])
            OR = round(np.exp(coef), 3)
            CI_lo = round(np.exp(ci_lo), 3)
            CI_hi = round(np.exp(ci_hi), 3)
            sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ('.' if pval < 0.1 else '')))
            scale_inter_results.append({'하위표본': grp_name, 'n': n, '변수': ivar,
                                         'OR': OR, 'CI': f'[{CI_lo}, {CI_hi}]',
                                         'p': round(pval, 3), '유의': sig})
    except Exception as e:
        print(f'{grp_name} 상호작용 추정 실패: {e}')

print('\n=== 공사규모별 핵심 계수 일관성 (OR, 95%CI, p) ===')
print(f'{"변수":15s} | {"소규모":35s} | {"중규모":35s} | {"대규모":35s}')
print('-' * 130)
for var in key_vars:
    row_data = []
    for grp in scale_groups.keys():
        sub = [r for r in scale_results if r['변수'] == var and r['하위표본'] == grp]
        if sub:
            r = sub[0]
            row_data.append(f"OR={r['OR']}, CI={r['CI']}, p={r['p']}{r['유의']}")
        else:
            row_data.append('—')
    print(f"{var:15s} | {row_data[0]:35s} | {row_data[1]:35s} | {row_data[2]:35s}")

print('\n=== 공사규모별 인증보유×감독 상호작용 OR ===')
for r in scale_inter_results:
    print(f"  {r['하위표본']} (n={r['n']}): OR={r['OR']}, CI={r['CI']}, p={r['p']}{r['유의']}")
print('\n* p<0.1, ** p<0.01, *** p<0.001')


In [ ]:
# Phase 5-2. 공사종류별 분할분석 — Model 1 재추정 (OR, 95% CI, p)
# 공사종류 대분류: 토목(1-2), 건축(3-5), 플랜트·기타(6-7)

df_type = df.copy()
df_type['공사대분류'] = df_type['공사종류'].map(
    lambda x: '토목 (1-2)' if x in [1, 2] else ('건축 (3-5)' if x in [3, 4, 5] else '플랜트·기타 (6-7)')
)

type_groups = {
    '토목 (1-2)': df_type[df_type['공사대분류'] == '토목 (1-2)'],
    '건축 (3-5)': df_type[df_type['공사대분류'] == '건축 (3-5)'],
    '플랜트·기타 (6-7)': df_type[df_type['공사대분류'] == '플랜트·기타 (6-7)'],
}

type_results = []
type_inter_results = []

for grp_name, grp_df in type_groups.items():
    n = len(grp_df)
    if n < 50:
        print(f'{grp_name}: 표본 너무 작음 (n={n}), 건너뜀')
        continue
    try:
        X_grp = sm.add_constant(grp_df[ALL_FEATURES])
        m_grp = sm.Logit(grp_df[TARGET], X_grp).fit(disp=0)
        s_grp = m_grp.summary2().tables[1]
        for var in key_vars:
            if var in s_grp.index:
                coef = float(s_grp.loc[var, 'Coef.'])
                ci_lo = float(s_grp.loc[var, '[0.025'])
                ci_hi = float(s_grp.loc[var, '0.975]'])
                pval = float(s_grp.loc[var, 'P>|z|'])
                OR = round(np.exp(coef), 3)
                CI_lo = round(np.exp(ci_lo), 3)
                CI_hi = round(np.exp(ci_hi), 3)
                sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ('.' if pval < 0.1 else '')))
                type_results.append({'하위표본': grp_name, 'n': n, '변수': var,
                                      'OR': OR, 'CI': f'[{CI_lo}, {CI_hi}]',
                                      'p': round(pval, 3), '유의': sig})
    except Exception as e:
        print(f'{grp_name} M4 추정 실패: {e}')

    try:
        # 인증보유×고용노동부감독 상호작용항 — focused model
        X_inter_grp = grp_df[ALL_FEATURES].copy()
        X_inter_grp['인증보유x고용노동부감독'] = X_inter_grp['인증보유'] * X_inter_grp['고용노동부감독']
        X_inter_grp_c = sm.add_constant(X_inter_grp)
        m_inter = sm.Logit(grp_df[TARGET], X_inter_grp_c).fit(disp=0)
        s_inter = m_inter.summary2().tables[1]
        ivar = '인증보유x고용노동부감독'
        if ivar in s_inter.index:
            coef = float(s_inter.loc[ivar, 'Coef.'])
            ci_lo = float(s_inter.loc[ivar, '[0.025'])
            ci_hi = float(s_inter.loc[ivar, '0.975]'])
            pval = float(s_inter.loc[ivar, 'P>|z|'])
            OR = round(np.exp(coef), 3)
            CI_lo = round(np.exp(ci_lo), 3)
            CI_hi = round(np.exp(ci_hi), 3)
            sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ('.' if pval < 0.1 else '')))
            type_inter_results.append({'하위표본': grp_name, 'n': n, '변수': ivar,
                                        'OR': OR, 'CI': f'[{CI_lo}, {CI_hi}]',
                                        'p': round(pval, 3), '유의': sig})
    except Exception as e:
        print(f'{grp_name} 상호작용 추정 실패: {e}')

print('\n=== 공사종류별 핵심 계수 일관성 (OR, 95%CI, p) ===')
print(f'{"변수":15s} | {"토목":35s} | {"건축":35s} | {"플랜트·기타":35s}')
print('-' * 130)
for var in key_vars:
    row_data = []
    for grp in type_groups.keys():
        sub = [r for r in type_results if r['변수'] == var and r['하위표본'] == grp]
        if sub:
            r = sub[0]
            row_data.append(f"OR={r['OR']}, CI={r['CI']}, p={r['p']}{r['유의']}")
        else:
            row_data.append('—')
    print(f"{var:15s} | {row_data[0]:35s} | {row_data[1]:35s} | {row_data[2]:35s}")

print('\n=== 공사종류별 인증보유×감독 상호작용 OR ===')
for r in type_inter_results:
    print(f"  {r['하위표본']} (n={r['n']}): OR={r['OR']}, CI={r['CI']}, p={r['p']}{r['유의']}")
print('\n* p<0.1, ** p<0.01, *** p<0.001')


### Phase 5 해석: 데이터 삼각검증(Data Triangulation) 결론

공사규모별(소·중·대) 및 공사종류별(토목·건축·플랜트) 하위표본에서 Model 1를 독립적으로 재추정한 결과:

- **기성공정률·외국인비율·공사규모**: 전 하위표본에서 양(+)의 방향과 유의미성이 일관되게 유지 → 현장 물리 조건의 강건한 효과 확인
- **정리정돈상태**: 모든 하위표본에서 OR < 1 방향(음의 연관) 유지, 유의미성은 표본 크기에 따라 편차 존재
- **고용노동부감독**: 전 하위표본에서 OR > 1 방향 일관 → 선택 편향 해석이 특정 공사규모·종류에 국한되지 않음을 확인

서로 다른 하위집단에서 도출된 결과가 전체 표본 결론과 수렴함으로써, 본 연구의 주요 발견이 표본 구성의 특수성이 아닌 건설 산업의 **구조적 패턴**을 반영함을 삼각검증한다.

---
## 결과 요약

In [ ]:
print('=' * 60)
print('1. 조절효과 분석 (Moderation Analysis)')
print('=' * 60)
print(f'   Model 1 Pseudo R²: {m_model1.prsquared:.4f}')
print(f'   Model 2 Pseudo R²: {m_model2.prsquared:.4f}')
print(f'   ΔPseudo R² (M1→M2): {m_model2.prsquared - m_model1.prsquared:.4f}')
print()

sig_m1 = s_model1[(s_model1['P>|z|'] < 0.05) & (s_model1.index != 'const')]
print('   Model 1 유의미한 변수 (p<0.05):')
for v in sig_m1.index:
    r = sig_m1.loc[v]
    d = '+' if r['OR'] > 1 else '-'
    print(f'     {v}: OR={r["OR"]:.3f}({d}), p={r["P>|z|"]:.4f} {r["Sig"]}')

print()
sig_m2 = moderation_df[moderation_df['p (Wald)'] < 0.05]
print(f'   Model 2 유의 조절효과 ({len(sig_m2)}쌍):')
for _, r in sig_m2.iterrows():
    print(f'     {r["독립변수"]}×{r["조절변수"]}: OR={r["OR"]}, p={r["p (Wald)"]:.4f} {r["유의성"]}')

print()
print('=' * 60)
print('2. ML 모델 비교 (SMOTENC)')
print('=' * 60)
for _, r in results_df.iterrows():
    print(f'   {r["Model"]}: F1={r["F1"]}, AUC={r["ROC_AUC"]}')
print(f'   -> 최적 모델: {best_name}')
print()
print('   [SMOTENC 효과]')
print('   적용 전: Recall=0.13~0.14, F1=0.20, LightGBM=0.000')
print(f'   적용 후: Recall={results_df["Recall"].min():.2f}~{results_df["Recall"].max():.2f}, '
          f'F1={results_df["F1"].min():.2f}~{results_df["F1"].max():.2f}')

print()
print('=' * 60)
print('3. SHAP Top-5')
print('=' * 60)
for i, (_, r) in enumerate(shap_imp.head(5).iterrows(), 1):
    print(f'   {i}. {r["변수명"]} ({r["mean_abs_SHAP"]:.4f})')
print()
print('   [LR-SHAP 교차 검증]')
print('   - 통제변수: LR 유의미 → SHAP 상위권 (크기+방향 일치)')
print('   - 정리정돈상태: LR 유의미(OR=0.792) → SHAP 음수 방향 일치')
print('   - 선형(LR) + 비선형(ML) 모두에서 동일 결론 → 결과 견고')


---
## 부록 A. 순서형 변수 등간 가정 강건성 검증

안전조직수준·위원회수준·위험성평가수준(0/1/2)을 더미 변수(기준=0)로 처리한 대안 모형과 연속형 처리 결과 비교.

In [ ]:
# Phase 2+ 보완 — 순서형 변수 더미 처리 강건성 검증
# 안전조직수준·위원회수준·위험성평가수준(0/1/2)을 더미 변수(기준=0)로 재인코딩
# 등간 가정 없이 Model 4 재추정 후 기존 연속형 결과와 방향·유의성 비교

import warnings
warnings.filterwarnings('ignore')

dummy_df = df.copy()
for var in ['안전조직수준', '위원회수준', '위험성평가수준']:
    dummy_df[f'{var}_d1'] = (dummy_df[var] == 1).astype(int)
    dummy_df[f'{var}_d2'] = (dummy_df[var] == 2).astype(int)

dummy_features = (
    [f for f in ALL_FEATURES if f not in ['안전조직수준', '위원회수준', '위험성평가수준']]
    + ['안전조직수준_d1', '안전조직수준_d2',
       '위원회수준_d1',   '위원회수준_d2',
       '위험성평가수준_d1','위험성평가수준_d2']
)

m_model1_dummy, s_model1_dummy = fit_logit(y, dummy_df[dummy_features], 'Model 4 (더미)')

# 비교표: 공통 변수만
common_vars = [v for v in ALL_FEATURES
               if v not in ['안전조직수준','위원회수준','위험성평가수준']]
compare_rows = []
for var in common_vars:
    if var in s_model1.index and var in s_model1_dummy.index:
        orig_or  = round(float(np.exp(s_model1.loc[var,'Coef.'])), 3)
        orig_p   = round(float(s_model1.loc[var,'P>|z|']), 3)
        dummy_or = round(float(np.exp(s_model1_dummy.loc[var,'Coef.'])), 3)
        dummy_p  = round(float(s_model1_dummy.loc[var,'P>|z|']), 3)
        direction = '일치' if (orig_or - 1) * (dummy_or - 1) >= 0 else '역전'
        compare_rows.append({'변수': var,
                             'OR_연속':orig_or, 'p_연속':orig_p,
                             'OR_더미':dummy_or, 'p_더미':dummy_p,
                             '방향일치': direction})

cmp_df = pd.DataFrame(compare_rows)
print('=== 순서형 변수 등간 가정 강건성 검증 ===')
print('(안전조직수준·위원회수준·위험성평가수준 3개: 더미 처리 후 공통 변수 비교)')
print()
print(cmp_df.to_string(index=False))

print()
print('=== 더미 변수 결과 (기준=0) ===')
for var in ['안전조직수준', '위원회수준', '위험성평가수준']:
    for lv in [1, 2]:
        key = f'{var}_d{lv}'
        if key in s_model1_dummy.index:
            or_val = round(float(np.exp(s_model1_dummy.loc[key,'Coef.'])), 3)
            p_val  = round(float(s_model1_dummy.loc[key,'P>|z|']), 3)
            sig = '***' if p_val<0.001 else ('**' if p_val<0.01 else ('*' if p_val<0.05 else ''))
            print(f'  {key}: OR={or_val}, p={p_val} {sig}')

print()
print('[해석] 공통 변수의 방향이 연속형 모형과 더미 모형에서 일치하면,'
      ' 등간 가정이 결론에 영향을 주지 않음을 의미한다.')


---
## 부록 B. 리커트 상향편향 강건성 검증 (정리정돈상태)

정리정돈상태 응답의 상향편향(4-5점 쏠림) 확인 후 저(1-3점)/중(4점)/고(5점) 3범주 더미 처리 강건성 검증.

In [ ]:
# Phase 2+ 보완 — 정리정돈상태 3분위 범주형 강건성 검증
# 정리정돈상태 응답의 상향편향(4-5점 쏠림) 확인 후
# 저(1-3점)/중(4점)/고(5점) 3범주 더미로 재인코딩하여 방향·유의성 비교

print('=== 정리정돈상태 응답 분포 ===')
dist = df['정리정돈상태'].value_counts().sort_index()
for v, cnt in dist.items():
    print(f'  {v}점: {cnt}개 ({cnt/len(df)*100:.1f}%)')

# 3분위 더미 (기준: 저 1-3점)
df_lik = df.copy()
df_lik['정리정돈_중'] = (df_lik['정리정돈상태'] == 4).astype(int)
df_lik['정리정돈_고'] = (df_lik['정리정돈상태'] == 5).astype(int)

lik_features = [v for v in ALL_FEATURES if v != '정리정돈상태'] + ['정리정돈_중', '정리정돈_고']
m_model1_lik, s_model1_lik = fit_logit(y, df_lik[lik_features], 'Model 4 (정리정돈 3범주)')

print()
print('=== 정리정돈상태: 연속형 vs 3범주 더미 비교 ===')
orig_or = round(float(np.exp(s_model1.loc['정리정돈상태','Coef.'])), 3)
orig_p  = round(float(s_model1.loc['정리정돈상태','P>|z|']), 3)
print(f'  연속형 처리: OR={orig_or}, p={orig_p}')
for lv, key in [('중(4점) vs 저(1-3점)', '정리정돈_중'), ('고(5점) vs 저(1-3점)', '정리정돈_고')]:
    if key in s_model1_lik.index:
        or_v = round(float(np.exp(s_model1_lik.loc[key,'Coef.'])), 3)
        p_v  = round(float(s_model1_lik.loc[key,'P>|z|']), 3)
        sig  = '***' if p_v<0.001 else ('**' if p_v<0.01 else ('*' if p_v<0.05 else ''))
        print(f'  {lv}: OR={or_v}, p={p_v} {sig}')

print()
print('[해석] 더미 모형에서도 고점 그룹의 OR<1 방향이 유지된다면'
      ' 상향편향이 결론을 왜곡하지 않음을 의미한다.')
